# Data Quality and Validation

Applies data quality rules to the cleaned customers, accounts and transactions, then routes the data.

**Reads:** `validate.bronze_clean_customers`, `validate.bronze_clean_accounts` and `validate.bronze_clean_transactions` (filtered by `BATCH_ID`)  
**Rules:** SQL expressions with `ERROR` or `WARN` severity. Rows with any `ERROR` are rejected.  
**Writes:** valid rows go to `silver.silver_customers`, `silver.silver_accounts` and `silver.silver_transactions` (merged after the first run). Rejected rows go to `Files/quarantine/DQ_Rejection/<entity>` as Parquet. A per-rule failure summary is appended to `dq_monitoring.dq_monitoring_table`.  
**Parameter:** `BATCH_ID`

### Imports

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### Shared functions

In [ ]:
%run NB_FUNCTIONS

### Parameters

The next cell is tagged `parameters`, so a pipeline can override it at run time.

In [ ]:
BATCH_ID = ""

## Customer

### Reading Only Latest Batch Data

In [ ]:
latest_cust_df = spark.read.table("validate.bronze_clean_customers")

In [ ]:
display(latest_cust_df)

### DQ Ingestion

In [ ]:
CUSTOMER_RULES = [
    {
        "name": "customer_id_format",
        "severity": "ERROR",
        "expr": "customer_id RLIKE '^CUST[0-9]{5}$'",
    },
    {
        "name": "dob_valid",
        "severity": "ERROR",
        "expr": "dob IS NOT NULL AND dob <= DATE'2025-07-01'",
    },
    {"name": "age_18_plus", "severity": "ERROR", "expr": "age >= 18"},
    {
        "name": "email_format",
        "severity": "WARN",
        "expr": "email RLIKE '^[a-z0-9._%+-]+@[a-z0-9-]+([.][a-z0-9-]+)*[.][a-z]{2,}$'",
    },
    {"name": "phone_valid", "severity": "WARN", "expr": "phone IS NOT NULL"},
    {
        "name": "pan_format",
        "severity": "WARN",
        "expr": "pan RLIKE '^[A-Z]{5}[0-9]{4}[A-Z]$'",
    },
    {
        "name": "pincode_format",
        "severity": "WARN",
        "expr": "pincode RLIKE '^[1-9][0-9]{5}$'",
    },
    {
        "name": "kyc_status_allowed",
        "severity": "WARN",
        "expr": "kyc_status IN ('VERIFIED', 'PENDING', 'REJECTED', 'EXPIRED')",
    },
    {"name": "income_valid", "severity": "WARN", "expr": "annual_income >= 0"},
]


latest_cust_DQ = apply_rules(latest_cust_df, CUSTOMER_RULES)

In [ ]:
customer_dq_count = (
    latest_cust_DQ.withColumn("issues", F.explode("dq_issues"))
    .groupBy("issues")
    .count()
)
display(customer_dq_count)

## Accounts

In [ ]:
latest_account_df = spark.read.table("validate.bronze_clean_accounts")
display(latest_account_df)

### DQ Ingestion

In [ ]:
ACCOUNT_RULES = [
    {
        "name": "account_type_known",
        "severity": "ERROR",
        "expr": "account_type <> 'UNKNOWN'",
    },
    {"name": "open_date_valid", "severity": "ERROR", "expr": "open_date IS NOT NULL"},
    {
        "name": "closed_after_open",
        "severity": "ERROR",
        "expr": "closed_date IS NULL OR closed_date >= open_date",
    },
    {
        "name": "closed_has_date",
        "severity": "WARN",
        "expr": "status <> 'CLOSED' OR closed_date IS NOT NULL",
    },
    {
        "name": "ifsc_format",
        "severity": "WARN",
        "expr": "ifsc_code RLIKE '^[A-Z]{4}0[A-Z0-9]{6}$'",
    },
    {
        "name": "limit_only_for_cards",
        "severity": "WARN",
        "expr": "(account_type = 'CREDIT_CARD') = (credit_limit IS NOT NULL)",
    },
    {
        "name": "card_luhn_valid",
        "severity": "ERROR",
        "expr": "account_type <> 'CREDIT_CARD' OR card_valid",
    },
    {"name": "currency_inr", "severity": "WARN", "expr": "currency = 'INR'"},
]

latest_account_DQ = apply_rules(latest_account_df, ACCOUNT_RULES)

### Orphan Account

In [ ]:
# Finding out account_id which doesn't exists
orphan_acc_df = latest_account_DQ.join(
    latest_cust_DQ.select("customer_id"), "customer_id", "left_anti"
)

print(f"Number of Orphan Account: {orphan_acc_df.count()}")

### Flagging Valid Customer_ID

In [ ]:
known_cust = latest_cust_DQ.select("customer_id", F.lit(True).alias("customers_exists"))

latest_account_DQ = (
    latest_account_DQ.drop("customers_exists")
    .join(F.broadcast(known_cust), "customer_id", "left")
    .withColumn(
        "dq_issues",
        F.when(
            F.col("customers_exists").isNull(),
            F.array_union(
                F.coalesce(F.col("dq_issues"), F.array().cast("array<string>")),
                F.array(F.lit("ERROR:customer_not_exists")),
            ),
        ).otherwise(F.col("dq_issues")),
    )
)

In [ ]:
account_dq_count = (
    latest_account_DQ.withColumn("issues", F.explode("dq_issues"))
    .groupBy("issues")
    .count()
)
display(account_dq_count)

## Transaction

In [ ]:
latest_transaction_df = spark.read.table("validate.bronze_clean_transactions").filter(
    F.col("_batch_id") == BATCH_ID
)
display(latest_transaction_df)

### Orphan Transaction

In [ ]:
# Orphaned Transaction (invalid AccountID transactions)

orphan_txn_df = latest_account_DQ.filter("account_id IS NOT NULL").join(
    latest_transaction_df.select("account_id"), "account_id", "left_anti"
)

print(f"Number of Orphan Transactions: {orphan_txn_df.count()}")

### DQ Ingestion

In [ ]:
# a debit on an account that was already closed is invalid; one before the account was opened is suspicious

account_attr = latest_account_df.select(
    "account_id",
    "customer_id",
    "account_type",
    F.col("status").alias("account_status"),
    "open_date",
    "closed_date",
    "credit_limit",
)

latest_transaction_df = latest_transaction_df.join(
    F.broadcast(account_attr), "account_id", "left"
)

TXN_RULES = [
    {
        "name": "account_id_present",
        "severity": "ERROR",
        "expr": "account_id IS NOT NULL",
    },
    {
        "name": "account_exists",
        "severity": "ERROR",
        "expr": "account_id IS NULL OR account_status IS NOT NULL",
    },
    {"name": "txn_ts_valid", "severity": "ERROR", "expr": "txn_ts IS NOT NULL"},
    {
        "name": "txn_not_in_future",
        "severity": "ERROR",
        "expr": "txn_ts IS NULL OR txn_ts < TIMESTAMP'2025-07-01 00:00:00+05:30'",
    },
    {"name": "amount_parsed", "severity": "ERROR", "expr": "amount IS NOT NULL"},
    {
        "name": "amount_positive",
        "severity": "ERROR",
        "expr": "amount IS NULL OR amount > 0",
    },
    {"name": "currency_known", "severity": "ERROR", "expr": "currency <> 'UNKNOWN'"},
    {"name": "txn_type_known", "severity": "ERROR", "expr": "txn_type <> 'UNKNOWN'"},
    {"name": "channel_known", "severity": "WARN", "expr": "channel <> 'UNKNOWN'"},
    {
        "name": "status_known",
        "severity": "ERROR",
        "expr": "status IN ('SUCCESS', 'FAILED', 'PENDING')",
    },
    {
        "name": "not_after_account_closed",
        "severity": "ERROR",
        "expr": "closed_date IS NULL OR txn_date_ist <= closed_date",
    },
    {
        "name": "not_before_account_open",
        "severity": "WARN",
        "expr": "open_date IS NULL OR txn_date_ist >= open_date",
    },
    {
        "name": "mcc_valid",
        "severity": "WARN",
        "expr": "NOT coalesce(mcc_invalid, false)",
    },
]

latest_transaction_DQ = apply_rules(latest_transaction_df, TXN_RULES)

In [ ]:
transaction_dq_count = (
    latest_transaction_DQ.withColumn("issues", F.explode("dq_issues"))
    .groupBy("issues")
    .count()
)
display(transaction_dq_count)

### Valid Refunds For Existing Transactions

In [ ]:
org_attr = latest_transaction_df.select(
    F.col("txn_id").alias("original_txn_id"),
    F.col("account_id").alias("orig_account_id"),
    F.col("amount").alias("orig_amount"),
    F.col("currency").alias("orig_currency"),
)

refund_status = (
    latest_transaction_DQ.filter(F.col("txn_type") == "REFUND")
    .join(org_attr, "original_txn_id", "left")
    .select(
        "txn_id",
        F.when(F.col("orig_amount").isNull(), "ERROR:refund_original_missing")
        .when(
            F.col("orig_account_id") != F.col("account_id"),
            "ERROR:refund_account_mismatch",
        )
        .when(F.col("amount") > F.col("orig_amount"), "ERROR:refund_exceeds_original")
        .alias("refund_issue"),
    )
    .filter("refund_issue IS NOT NULL")
)

refund_status.groupBy("refund_issue").count().show(truncate=False)

latest_transaction_DQ = (
    latest_transaction_DQ.join(refund_status, "txn_id", "left")
    .withColumn(
        "dq_issues",
        F.when(
            F.col("refund_issue").isNotNull(),
            F.array_union("dq_issues", F.array("refund_issue")),
        ).otherwise(F.col("dq_issues")),
    )
    .drop("refund_issue")
)

### Routing Data to Silver and Quarantine

In [ ]:
cust_valid = routing(
    df=latest_cust_DQ,
    entity="customers",
    schema="silver",
    table_name="silver_customers",
    merge_condition="target.customer_id = source.customer_id",
    match_condition="source.updated_at > target.updated_at",
)

acc_valid = routing(
    df=latest_account_DQ,
    entity="accounts",
    schema="silver",
    table_name="silver_accounts",
    merge_condition="target.account_id = source.account_id",
    match_condition="source.updated_at > target.updated_at",
)

txn_valid = routing(
    df=latest_transaction_DQ,
    entity="transactions",
    schema="silver",
    table_name="silver_transactions",
    merge_condition="target.txn_id = source.txn_id",
    match_condition="source.updated_at > target.updated_at",
).cache()


print(f"Number of Valid Customer Records:    {cust_valid.count()}")
print(f"Number of Valid Account Records:     {acc_valid.count()}")
print(f"Number of Valid Transaction Records: {txn_valid.count()}")

### Reporting to DQ Monitoring Dashboard

In [ ]:
# DQ Dashboard
dq_report = (
    dq_summary(latest_cust_DQ, "customers", BATCH_ID)
    .unionByName(dq_summary(latest_account_DQ, "accounts", BATCH_ID))
    .unionByName(dq_summary(latest_transaction_DQ, "transactions", BATCH_ID))
    .orderBy(F.col("entity").asc(), F.desc("failed_rows"))
)
dq_report.show(40, truncate=False)

# create schema for DQ Monitoring Dashboard
spark.sql("CREATE SCHEMA IF NOT EXISTS dq_monitoring")
dq_report.coalesce(1).write.mode("append").saveAsTable(
    "dq_monitoring.dq_monitoring_table"
)